# 01 — Análise Exploratória (Parte 0)

Entender o dado bruto antes de transformar qualquer coisa.

**Este notebook não reimplementa nada.** Toda a lógica vive em `src/mp/analysis/` e
volta a rodar na ingestão em produção (Parte 2). Aqui só chamamos e discutimos.

**Pronto quando** existirem: a lista de colunas a descartar, a lista de rótulos e a
tabela de assinaturas por rótulo para cruzar com os PDFs.

In [ ]:
import sys
from pathlib import Path

# O pacote ainda não está instalado; aponta para src/ na mão.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from mp import config
from mp.analysis import (
    carregar, colunas_numericas,
    resumo_geral, nulos_por_coluna, perfil_rotulos, janela_temporal,
    taxa_amostragem, sugerir_familias,
    colunas_constantes, colunas_redundantes, duplicatas_consecutivas, outliers_iqr,
    assinaturas_por_rotulo, assinatura_de_rotulo, comparar_com_global,
    colunas_a_descartar,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("CSV:", config.caminho_csv())

In [ ]:
df = carregar()
resumo_geral(df)

## 1. Nulos

166.796 linhas × 26 colunas, **zero nulos declarados**.

Ausência de `NaN` não é ausência de dado faltante: um sensor sem leitura pode ter sido
gravado como `0.0`. Esse candidato aparece nas seções de constantes e de outliers, não aqui.

A coluna `distintos` já adianta um achado: `rpm` tem 5 valores em 166 mil linhas.

In [ ]:
nulos_por_coluna(df)

## 2. Janela temporal e taxa de amostragem

**Achado que muda decisão de engenharia:** `created_at` **não é monotônico**.
O arquivo tem saltos negativos de dezenas de dias — são sessões gravadas em épocas
diferentes e concatenadas fora de ordem.

Consequência direta: a mediana móvel da Parte 3 e a formação de episódios da Parte 1
precisam **ordenar por `created_at` antes** e nunca atravessar a fronteira entre sessões.
Por isso `taxa_amostragem` ordena internamente — sem isso os intervalos vêm negativos.

In [ ]:
janela_temporal(df)

In [ ]:
amostragem = taxa_amostragem(df)
{k: v for k, v in amostragem.items() if k != "intervalos"}

O intervalo mediano é **2,0 s**, confirmando a expectativa. 92% das leituras respeitam
essa cadência; o resto se concentra num segundo modo perto de 5,3 s — provavelmente
outra configuração de datalogger em parte das sessões.

Os 330 cortes acima de 60 s estimam **331 sessões** de coleta. Esse número é o insumo
para a validação por grupo da Parte 3: segurar a sessão inteira fora, não leituras
individuais, senão a autocorrelação temporal infla a acurácia.

In [ ]:
s = pd.Series(amostragem["intervalos"])
s[s <= 10].plot.hist(bins=60, logy=True, figsize=(10, 3),
                     title="Intervalo entre leituras (s) — eixo Y em log");

## 3. Rótulos

**Segundo achado:** são **151 valores distintos** em `fault`, não ~10.

A inflação vem de três fontes:

1. **Erros de digitação** — `mortor_desligado_novo`, `normla_carga_3_3`,
   `cockecocked_adxl_0`, `desabalanceado_3`, `ddesbalanceado_adxl_0`
2. **Sufixos de sessão/montagem** — `_2`, `_3`, `_pos_2`, `_carga`, `_adxl_0`, `_novo`
3. **Prefixo `new_`** — nova campanha de coleta, mesmo defeito

A coluna `span_horas` mostra por que contar linhas engana: `rolamento_inner` tem 13.000
leituras concentradas em 34 h. É **uma sessão contínua**, não 13.000 ocorrências.
Colapsar isso em episódios é o primeiro item da Parte 1.

In [ ]:
rotulos = perfil_rotulos(df)
print(f"{len(rotulos)} rótulos | {int(rotulos['e_problema'].sum())} defeitos, "
      f"{int((~rotulos['e_problema']).sum())} estados")
rotulos.head(20)

In [ ]:
# Agrupamento SUGERIDO. A decisão final é curada à mão no data/fault_map.yaml (Parte 1).
familias = sugerir_familias(df[config.COLUNA_ROTULO].dropna().unique())
familias["familia_sugerida"].value_counts()

In [ ]:
# Rótulos que o heurístico não classificou — é exatamente onde o curador deve olhar.
familias[familias["familia_sugerida"] == "?"]

As 16 famílias sugeridas cobrem os 151 rótulos sem sobra. Seis delas correspondem aos
PDFs disponíveis (desalinhamento, desbalanceamento, cocked rotor, rolamentos, polias,
correias). As demais — `eccentric_rotor`, `ventoinha`, `falta_fase` — **não têm
documento**, e é justamente o guardrail G3 que vai recusar prescrição para elas.
Esse mapeamento vira o `fault_map.yaml`.

## 4. Qualidade: constantes, redundâncias e duplicatas

**Terceiro achado — este contraria o GUIA.md.** A suspeita registrada era de que
`z_peak_vel_comp_freq_hz` e `x_peak_vel_comp_freq_hz` fossem constantes em 61 Hz.
**Não são:** têm 79 e 50 valores distintos. 61 Hz é a moda (60% e 49% das linhas),
não o valor único.

Elas carregam informação e **não devem ser descartadas** — a frequência do pico se
desloca justamente em alguns defeitos, que é o sinal que queremos.

In [ ]:
colunas_constantes(df).head(8)

`rpm` tem 5 valores (0, 500, 1000, 2000, 3000): é uma **categórica de regime disfarçada
de numérica**. Importa para o kNN da Parte 3 — comparar 500 com 3000 como distância
contínua não significa nada físico.

### Unidades duplicadas — confirmado

Testamos a identidade numérica, não a correlação. O erro máximo fica na casa do
arredondamento do arquivo, o que prova que uma coluna é conversão da outra.

In [ ]:
colunas_redundantes(df)

Os 5 pares são redundantes. Mantemos o SI (mm/s, °C) e descartamos a versão imperial:
duplicar a mesma grandeza faz o `StandardScaler` contá-la duas vezes e infla o peso
dela na distância do kNN.

### Duplicatas consecutivas

In [ ]:
dup = duplicatas_consecutivas(df)
print(f"{dup['total']} linhas idênticas à anterior ({dup['pct']}%)")
dup["por_rotulo"].head(10)

5,8% das linhas repetem a anterior em **todas** as colunas de medida. A comparação
ignora `id` e `created_at` de propósito — eles sempre mudam.

Duas leituras iguais em 4 casas decimais a 2 s de distância são, quase certamente, a
mesma amostra repetida pelo datalogger. Na Parte 3 elas viram vizinhos de distância
zero que não acrescentam informação. Deduplicar é item da Parte 1.

## 5. Outliers — identificados, não tratados

Critério de Tukey (IQR), não z-score: várias colunas são fortemente assimétricas
(`z_kurtosis` tem mediana 2,5 e máximo 65), e a média/desvio que o z-score usa já
estão contaminados pelos próprios extremos que deveriam detectar.

**Nada é removido.** Em vibração o pico raro costuma ser o sinal: kurtosis alta é
exatamente a assinatura de impacto de rolamento. Descartar por regra estatística
apagaria a falha que o sistema existe para detectar.

In [ ]:
outliers_iqr(df).head(12)

Duas leituras diferentes na mesma tabela:

- **`% outliers` alto + `max_sobre_limite` ≈ 1** — dispersão larga e uniforme.
  `z_peak_vel_comp_freq_hz` (25% fora dos limites) é o caso: o IQR é 2,5 Hz porque a
  massa está grudada em 61 Hz, então qualquer desvio legítimo vira "outlier".
  Artefato do critério, não anomalia.
- **`% outliers` baixo + `max_sobre_limite` nas dezenas** — cauda longa de impacto.
  `z_peak_acceleration_g` chega a **49×** o limite superior. Poucos eventos, muito
  acima do normal. **Este é o sinal.**

Recalcular os limites dentro de cada rótulo separa a variação natural da classe do
pico que destoa da própria classe:

In [ ]:
outliers_iqr(df[df[config.COLUNA_ROTULO] == "rolamento_inner"]).head(6)

## 6. Assinaturas por rótulo — entrega da Parte 0

Mediana, não média: kurtosis e crest factor são definidos sobre picos, e um impacto
isolado desloca a média do rótulo inteiro.

É esta tabela que será cruzada com o que os PDFs descrevem. **Divergência entre o
medido e o documentado é achado, não erro.**

In [ ]:
assinaturas = assinaturas_por_rotulo(df, min_leituras=100)
print(assinaturas.shape)
assinaturas.head(15)

In [ ]:
# O que distingue um rótulo específico do resto do dataset.
comparar_com_global(df, "rolamento_inner").head(10)

In [ ]:
# `cv` alto = classe dispersa = classe que o kNN vai confundir na Parte 3.
assinatura_de_rotulo(df, "rolamento_inner")

### Ponto de atenção para a Parte 1

Os desvios entre rótulos são **pequenos nas medianas** — poucos pontos percentuais em
relação à mediana global. Isso tem duas leituras possíveis:

1. As classes se distinguem pela **cauda** (picos de impacto), não pelo centro — o que
   reforça manter kurtosis, crest factor e as acelerações de pico como features;
2. Parte dos rótulos são a mesma condição física sob nomes diferentes — o que o
   `fault_map.yaml` vai consolidar.

A validação por grupo da Parte 3 é que vai separar as duas hipóteses. Se a acurácia
cair muito ao segurar a sessão inteira fora, o modelo estava reconhecendo a **sessão**,
não o defeito.

## 7. Decisão consolidada: colunas a descartar

In [ ]:
colunas_a_descartar(df)

**Descartar (6):** as 4 colunas `*_in_s`, `temperature_f`, e os identificadores `id` /
`created_at` como *features* (continuam no banco como metadado).

**Manter, apesar da suspeita inicial:**

| Coluna | Por quê |
|---|---|
| `z_peak_vel_comp_freq_hz`, `x_peak_vel_comp_freq_hz` | Não são constantes — 79 e 50 valores distintos |
| `rpm` | Mantida, mas tratada como **categórica de regime** (5 patamares) |

**Restam 17 features numéricas** para o motor de similaridade da Parte 3.

In [ ]:
descartar = set(colunas_a_descartar(df)["coluna"])
features = [c for c in colunas_numericas(df) if c not in descartar]
print(len(features), "features:")
features

---

## Checklist da Parte 0

- [x] **Lista de colunas a descartar** — seção 7, com o motivo verificado de cada uma
- [x] **Lista de rótulos** — seção 3, com a família sugerida e a marcação defeito/estado
- [x] **Tabela de assinaturas por rótulo** — seção 6, pronta para cruzar com os PDFs

### Entra na Parte 1

1. Aplicar os 6 descartes
2. Deduplicar as 9.736 leituras consecutivas idênticas
3. Colapsar em episódios usando o corte de 60 s (`config.GAP_NOVA_SESSAO_S`)
4. Escrever o `fault_map.yaml` consolidando os 151 rótulos nas 16 famílias
5. Cruzar as assinaturas com os 6 PDFs e registrar as divergências